In [39]:
import pandas as pd
pd.set_option('display.max_colwidth', None)



In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Set up Chrome options
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--ignore-certificate-errors")

# Start the browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the player futures page
url = "https://www.bettingpros.com/nfl/odds/player-futures/"
driver.get(url)

# Wait until at least one player card link appears
try:
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a.player-card__link"))
    )
except:
    print("❌ Player links didn’t load in time.")


link_elements = driver.find_elements(By.CSS_SELECTOR, "a.player-card__link")
print(f"✅ Found {len(link_elements)} player links")

# Print first 5 full URLs
for link in link_elements[:5]:
    href = link.get_attribute("href")
    print(f"https://www.bettingpros.com{href}")

player_slugs = []
for link in link_elements:
    href = link.get_attribute("href")  # Get the actual URL string from the WebElement
    slug = href.split('/')[-2]  # Extract slug from URL string
    player_slugs.append(slug)

print(f"Found {len(player_slugs)} player slugs")
print(f"First 5 players: {player_slugs[:5]}")

driver.quit()


✅ Found 687 player links
https://www.bettingpros.comhttps://www.bettingpros.com/nfl/odds/player-futures/jayden-daniels/
https://www.bettingpros.comhttps://www.bettingpros.com/nfl/odds/player-futures/jalen-hurts/
https://www.bettingpros.comhttps://www.bettingpros.com/nfl/odds/player-futures/josh-allen-qb/
https://www.bettingpros.comhttps://www.bettingpros.com/nfl/odds/player-futures/justin-fields/
https://www.bettingpros.comhttps://www.bettingpros.com/nfl/odds/player-futures/travis-hunter/
Found 687 player slugs
First 5 players: ['jayden-daniels', 'jalen-hurts', 'josh-allen-qb', 'justin-fields', 'travis-hunter']


In [51]:
import requests
import time

all_players_data = []

for player_slug in player_slugs[200:]:  # Start with first 10 players for testing
    print(f"\n--- Processing {player_slug} ---")

    url = "https://api.bettingpros.com/v3/offers"
    params = {
        "sport": "NFL",                     # Sport type
        "market_id": "4:5:6:7:22:23:47:48:49:50:51:52:53:54:55:56:57:77:125:300:301:302:303:304:305:306:33",  # All the stat categories we're interested in
        "player_slug": player_slug,       # The player we're requesting stats for
        "season": 2025,                     # The NFL season year
        "book_id": "null",                  # Leave this as null to include all sportsbooks
        "limit": 5,                         # We want 5 results per request (the API's default)
        "page": 1                           # We'll change this as we loop to get all pages
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://www.bettingpros.com/",
        "Origin": "https://www.bettingpros.com",
        "x-api-key": "CHi8Hy5CEE4khd46XNYL23dCFX96oUdw6qOt1Dnh",
    }

    # Step 4: Prepare a list to store all the results we get from each page
    player_results = []
    # The loop is continuing forever becauase the data is returning rempty lists
    # Soolution is to track unique offers by ID (or use repr(offer) if no ID)
    seen_offer_ids = set()  


    # Step 5: Loop through pages until there are no more results
    while True:
        # Send the request
        response = requests.get(url, headers=headers, params=params)
        
        debug_url = response.url  # For debugging, see the full URL being requested
        print(f"Requesting: {debug_url}")
        # Check if the request was successful (status code 200)
        if response.status_code != 200:
            print(f"Error on page {params['page']}: {response.status_code}")
            break  # Stop the loop if there's an error

        # Convert the response to JSON (Python dictionary)
        data = response.json()
        offers = data.get("offers", [])
        # Get the list of "offers" from the response
        new_offers = []
        for offer in offers:
            offer_key = offer["id"]
            print(f"Processing offer ID: {offer_key} on page {params['page']}")
            if offer_key not in seen_offer_ids:
                seen_offer_ids.add(offer_key)
                new_offers.append(offer)


        # If we got no offers, it means we’re out of pages — stop the loop
        if not new_offers:
            break

        # Add the offers from this page to our master list
        player_results.extend(new_offers)
        
        # Print a message so we know it's working
        print(f"Fetched page {params['page']} with {len(offers)} offers.")

        # Move to the next page
        params["page"] += 1
        # Give the page a second beore the next request
        time.sleep(1)

    all_players_data.extend(player_results)
    print(f"Got {len(player_results)} offers for {player_slug}")

# Step 4: Save to CSV
df= pd.DataFrame(all_players_data)


--- Processing demario-douglas ---
Requesting: https://api.bettingpros.com/v3/offers?sport=NFL&market_id=4%3A5%3A6%3A7%3A22%3A23%3A47%3A48%3A49%3A50%3A51%3A52%3A53%3A54%3A55%3A56%3A57%3A77%3A125%3A300%3A301%3A302%3A303%3A304%3A305%3A306%3A33&player_slug=demario-douglas&season=2025&book_id=null&limit=5&page=1
Processing offer ID: 08c35e7e-bc0f-4bd4-b547-d0a4beef83c6 on page 1
Processing offer ID: 786bb1d3ce114f022af1ed1e2f64cfb1 on page 1
Processing offer ID: 91055e89-08f2-4196-a818-20568fe66913 on page 1
Processing offer ID: fd7df31e780e7bb8f4e7eb79d6097bc1 on page 1
Fetched page 1 with 4 offers.
Requesting: https://api.bettingpros.com/v3/offers?sport=NFL&market_id=4%3A5%3A6%3A7%3A22%3A23%3A47%3A48%3A49%3A50%3A51%3A52%3A53%3A54%3A55%3A56%3A57%3A77%3A125%3A300%3A301%3A302%3A303%3A304%3A305%3A306%3A33&player_slug=demario-douglas&season=2025&book_id=null&limit=5&page=2
Processing offer ID: 08c35e7e-bc0f-4bd4-b547-d0a4beef83c6 on page 2
Processing offer ID: 786bb1d3ce114f022af1ed1e2f64cfb

In [52]:
import time
from datetime import datetime

# Step 2 – Add new empty columns to hold the values we’ll extract
extracted_data = []

for idx, row in df.iterrows():
    market_id = row['market_id']
    #apply the market_id mapping at the end
    participants = row["participants"]

    # Safely get player info
    player_name = participants[0].get("name", "") if participants else ""
    player_info = participants[0].get("player", {}) if participants else {}
    position = player_info.get("position", "")
    team = player_info.get("team", "")

    for selection in row["selections"]:
        opening_line = selection.get("opening_line",{})
        opening_cost = opening_line.get("cost","")
        
        for book in selection.get("books", []):
            book_id = book.get("id")
            for line in book.get("lines", []):
                if line.get("best") == True:
                    best_cost = line.get("cost","")
                    best_line = line.get("line","")
                    break # Check until best line and then break
    # print(market_id, player_name, position, team, opening_cost, best_cost, best_line)
    extracted_data.append({
        "market_id": market_id,
        "player_name": player_name,
        "position": position,
        #"team": team,
        #"opening_cost": opening_cost,
        #"best_cost": best_cost,
        "best_line": best_line
    })

# Create new DataFrame from the list
scraped_df = pd.DataFrame(extracted_data)


In [53]:
market_mapping = {
    4: "Comeback Player of the Year",
    5: "Season MVP",
    6: "Defensive Player of the Year",
    7: "Offensive Player of the Year",
    22:"Defensive Rookie of the Year",
    23:"Offensive Rookie of the Year", 
    47: "Most Interceptions",
    49: "Most Passing TDs",
    50: "Most Passing Yards",
    51: "Most Receiving TDs",
    52: "Most Receiving Yards",
    53: "Most Receptions",
    54: "Most Rushing TDs",
    55: "Most Rushing Yards",
    77: "Super Bowl MVP",
    300: "Total Passing Yards",
    301: "Total Rushing Yards",
    302: "Total Receiving Yards",
    303: "Total Interceptions",
    304: "Total Passing TDs",
    305: "Total Rushing TDs",
    306: "Total Receiving TDs"
}

In [54]:

# Apply the market mapping
scraped_df['market_name'] = scraped_df['market_id'].map(market_mapping)
scraped_df = scraped_df.drop(columns=['market_id'])  # Drop the market_id column if not needed
 # Drop the market_id column if not needed
scraped_df.head()


,player_name,position,best_line,market_name
0,DeMario Douglas,WR,1.0,Offensive Player of the Year
1,DeMario Douglas,WR,1.0,Most Receiving TDs
2,DeMario Douglas,WR,1.0,Most Receiving Yards
3,DeMario Douglas,WR,1.0,Most Receptions
4,Austin Ekeler,RB,1.0,Offensive Player of the Year


In [55]:
date = datetime.now().strftime("%Y-%m-%d_%H%M%S")

df_wide = scraped_df.copy()


In [56]:

valid_positions = ['QB', 'RB', 'WR', 'TE']
df_wide = df_wide[df_wide['position'].isin(valid_positions)]


In [57]:

df_wide = df_wide.pivot(index=['player_name', 'position'],
                columns='market_name', 
                values='best_line').reset_index()

df_wide.head()

market_name,player_name,position,NaN,Comeback Player of the Year,Defensive Player of the Year,Most Interceptions,Most Passing TDs,Most Passing Yards,Most Receiving TDs,Most Receiving Yards,Most Receptions,Most Rushing TDs,Most Rushing Yards,Offensive Player of the Year,Offensive Rookie of the Year,Season MVP,Super Bowl MVP,Total Receiving TDs,Total Receiving Yards,Total Rushing Yards
0,A.J. Dillon,RB,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Adonai Mitchell,WR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alec Pierce,WR,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,Alexander Mattison,RB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Allen Lazard,WR,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
for col in df_wide.columns:
    if col in ['market_id',
            'Total Interceptions',
            'Super Bowl MVP',
            'Comeback Player of the Year',
            'Defensive Player of the Year',
            'Defensive Rookie of the Year',
            'Most Interceptions',
            'Most Passing TDs',
            'Most Passing Yards',
            'Most Receiving TDs',
            'Most Receiving Yards',
            'Most Receptions',
            'Most Rushing TDs',
            'Most Rushing Yards',
            'Offensive Player of the Year',
            'Offensive Rookie of the Year',
            'Season MVP']:
                df_wide=df_wide.drop(columns=[col])

df_wide.to_csv(f"player_props_2025_{date}.csv", index=False)
print(f"Saved to player_props_2025_{date}.csv")



Saved to player_props_2025_2025-07-28_200925.csv


: 

In [ ]:
    # Step 4 – Assign values back into the DataFrame
